# mixup — mixup: Beyond Empirical Risk Minimization (2017)

_Papers · Foundations & Optimization_

**Train on blended pairs of examples: feed the network a weighted average of two inputs and the same weighted average of their labels, and it generalizes better and behaves more smoothly between classes.**

---

This notebook is a practice scaffold for the **mixup — mixup: Beyond Empirical Risk Minimization (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build mixup in three moves: confirm on one hand-blended example that the mixed loss equals the `lambda`-weighted average of the two per-class losses, write the few-line mixup transform that blends inputs **and** labels, then train the same tiny classifier with and without mixup and sweep the strength `alpha`.

### Step 1 — Blend one pair by hand

Take one example from each class, pick `lambda = 0.7`, and form the mixed input and the mixed (soft) label with the **same** lambda (Section 2). We then check the key identity: the cross-entropy of a prediction against the soft label equals `lambda*CE0 + (1-lambda)*CE1`, the lambda-weighted average of the two single-class losses.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

# One example per class.
xi = torch.tensor([0., 2.])
yi = torch.tensor([1., 0.])   # class 0
xj = torch.tensor([3., 1.])
yj = torch.tensor([0., 1.])   # class 1
lam = 0.7

# Blend the input AND the label with the same lambda.
x_tilde = lam * xi + (1 - lam) * xj
y_tilde = lam * yi + (1 - lam) * yj
print("x~ =", x_tilde.tolist(), " y~ =", y_tilde.tolist())   # [0.9,1.7] [0.7,0.3]

# Mixed cross-entropy equals the lambda-weighted average of the two per-class losses.
p = torch.tensor([0.6, 0.4])                  # model's predicted probs on x~
mixed_ce = -(y_tilde * p.log()).sum()
ce0 = -p[0].log()
ce1 = -p[1].log()
weighted = lam * ce0 + (1 - lam) * ce1
print("mixed CE =", round(mixed_ce.item(), 4),
      " lam*CE0+(1-lam)*CE1 =", round(weighted.item(), 4))   # both 0.6325

### Step 2 — The mixup transform and a toy problem

`mixup_batch` draws one `lambda` from a Beta(`alpha`, `alpha`) distribution, randomly re-pairs the batch, and returns the blended inputs and blended labels — mixing the label too is the whole point. We pair it with a tiny two-class problem (a small train set leaves room for mixup to help) and a soft cross-entropy loss that accepts these blended targets.

In [ ]:
# mixup FROM SCRATCH (the few-line transform).
def mixup_batch(x, y, alpha):
    lam = float(np.random.beta(alpha, alpha))   # one lambda per batch
    idx = torch.randperm(x.size(0))             # random pairing within the batch
    x_mix = lam * x + (1 - lam) * x[idx]
    y_mix = lam * y + (1 - lam) * y[idx]        # mix the LABEL too -- the whole point
    return x_mix, y_mix

# Toy 2-class problem: tiny train set, large test set.
def make_data(n, seed):
    r = np.random.default_rng(seed)
    t0 = r.uniform(0, np.pi, n)
    X0 = np.stack([np.cos(t0), np.sin(t0)], 1) + r.normal(0, .18, (n, 2))
    t1 = r.uniform(0, np.pi, n)
    X1 = np.stack([1 - np.cos(t1), .4 - np.sin(t1)], 1) + r.normal(0, .18, (n, 2))
    X = np.vstack([X0, X1])
    y = np.array([0] * n + [1] * n)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y)

Xtr, ytr = make_data(30, 1)      # small -> room for mixup to help
Xte, yte = make_data(400, 99)
Ytr = F.one_hot(ytr, 2).float()

def make_net():
    return nn.Sequential(
        nn.Linear(2, 16), nn.ReLU(),
        nn.Linear(16, 16), nn.ReLU(),
        nn.Linear(16, 2),
    )

def soft_ce(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, 1)).sum(1).mean()

### Step 3 — Train ERM vs mixup and sweep alpha

With everything else held fixed — same network, data, and seed — we train once with plain ERM and once with mixup, then report train/test accuracy and the train-test gap. Finally we sweep `alpha` over a range to see how the mixing strength trades off generalization.

In [ ]:
def train(use_mixup, alpha=0.2, steps=4000, lr=0.3, seed=1):
    torch.manual_seed(seed)
    np.random.seed(seed)
    net = make_net()
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    for _ in range(steps):
        if use_mixup:
            xb, yb = mixup_batch(Xtr, Ytr, alpha)
        else:
            xb, yb = Xtr, Ytr
        opt.zero_grad()
        soft_ce(net(xb), yb).backward()
        opt.step()
    return net

@torch.no_grad()
def report(net, tag):
    tr = (net(Xtr).argmax(1) == ytr).float().mean().item()
    pte = F.softmax(net(Xte), 1)
    te = (pte.argmax(1) == yte).float().mean().item()
    mean_conf = pte.max(1).values.mean().item()
    print(f"{tag:14s} train={tr:.3f} test={te:.3f} gap={tr-te:+.3f} "
          f"mean_conf={mean_conf:.3f}")

print("\n--- ERM vs mixup (same net, data, seed) ---")
report(train(False), "ERM")
report(train(True, alpha=0.2), "mixup a=0.2")

print("\n--- alpha ablation (test accuracy) ---")
report(train(False), "ERM (no mixup)")
for a in [0.1, 0.2, 0.4, 1.0, 4.0]:
    report(train(True, alpha=a), f"mixup a={a}")

## Visualize the data & results

_Does training the SAME tiny classifier with mixup (vs plain ERM) generalize better on a small toy 2-class problem — and how does the mixup strength alpha trade off?_

### Step A — Rebuild data, model, and training

We reconstruct the toy problem and training loop so the comparison is self-contained: the same two-arc dataset, the same small MLP, and a `train` that optionally applies the mixup transform. This sets up the apples-to-apples ERM-vs-mixup comparison that follows.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def make_data(n, seed):
    r = np.random.default_rng(seed)
    t0 = r.uniform(0, np.pi, n)
    X0 = np.stack([np.cos(t0), np.sin(t0)], 1) + r.normal(0, .18, (n, 2))
    t1 = r.uniform(0, np.pi, n)
    X1 = np.stack([1 - np.cos(t1), .4 - np.sin(t1)], 1) + r.normal(0, .18, (n, 2))
    return (torch.tensor(np.vstack([X0, X1]), dtype=torch.float32),
            torch.tensor(np.array([0] * n + [1] * n)))

Xtr, ytr = make_data(30, 1)
Xte, yte = make_data(400, 99)
Ytr = F.one_hot(ytr, 2).float()

def mixup_batch(x, y, a):
    lam = float(np.random.beta(a, a))
    idx = torch.randperm(x.size(0))
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]

def train(use_mixup, alpha=0.2, steps=4000, lr=0.3, seed=1):
    torch.manual_seed(seed)
    np.random.seed(seed)
    net = nn.Sequential(
        nn.Linear(2, 16), nn.ReLU(),
        nn.Linear(16, 16), nn.ReLU(),
        nn.Linear(16, 2),
    )
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    for _ in range(steps):
        if use_mixup:
            xb, yb = mixup_batch(Xtr, Ytr, alpha)
        else:
            xb, yb = Xtr, Ytr
        opt.zero_grad()
        loss = -(yb * F.log_softmax(net(xb), 1)).sum(1).mean()
        loss.backward()
        opt.step()
    return net

### Step B — Compare ERM vs mixup and sweep alpha

We report train/test accuracy for plain ERM and for mixup at `alpha = 0.2`, then sweep `alpha` to trace the trade-off. Expect an inverted-U: mixup beats ERM across a wide range, peaking at small-to-moderate `alpha`, then degrading as the blends grow too aggressive.

In [ ]:
@torch.no_grad()
def stats(net):
    tr = (net(Xtr).argmax(1) == ytr).float().mean().item()
    te = (net(Xte).argmax(1) == yte).float().mean().item()
    return round(tr, 4), round(te, 4)

print("ERM   ", stats(train(False)))            # (1.0, 0.935)
print("mixup ", stats(train(True, alpha=0.2)))  # (1.0, 0.97)
print("alpha sweep test acc:")
print(" ERM ", stats(train(False))[1])
for a in [0.1, 0.2, 0.4, 1.0, 4.0]:
    print(f" a={a}", stats(train(True, alpha=a))[1])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Mix example $i$ = (input $[1,4]$, label class 0 = $[1,0]$) with example $j$ = (input $[5,0]$, label class 1 = $[0,1]$) using $\lambda=0.25$. Give $\tilde{x}$ and $\tilde{y}$, and say which class the soft label leans toward.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Blend the input: $\tilde{x}=0.25[1,4]+0.75[5,0]$. — _Same convex combination applied componentwise._
- Compute: $[0.25\cdot1+0.75\cdot5,\;0.25\cdot4+0.75\cdot0]=[4.0,\,1.0]$. — _Weighted average of the two inputs._
- Blend the label: $\tilde{y}=0.25[1,0]+0.75[0,1]=[0.25,\,0.75]$. — _The label uses the SAME $\lambda$._

**Answer:** $\tilde{x}=[4.0,\,1.0]$ and $\tilde{y}=[0.25,\,0.75]$. With $\lambda=0.25$ the blend is mostly example $j$, so the soft label leans toward class 1 (0.75).

</details>

**Problem 2.** With the soft label $\tilde{y}=[0.7,0.3]$ from the worked example, a model predicts $p=[0.9,0.1]$. Compute the mixed cross-entropy loss. Is it smaller or larger than against $p=[0.6,0.4]$ (which gave 0.6325)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mixed CE $=-(0.7\ln 0.9+0.3\ln 0.1)$. — _Cross-entropy against the soft target._
- $-\ln 0.9=0.1054$, $-\ln 0.1=2.3026$. — _Per-class losses._
- $0.7(0.1054)+0.3(2.3026)=0.0738+0.6908=0.7646$. — _$\lambda$-weighted sum._

**Answer:** Loss $\approx 0.7646$ — larger than 0.6325. Even though $p=[0.9,0.1]$ is more confident about class 0, the soft target wants $0.3$ mass on class 1; over-committing to class 0 is penalized. This is exactly how mixup discourages overconfidence.

</details>

**Problem 3.** Ablation: in the CODE, sweep $\alpha\in\{0.1, 0.2, 0.4, 1.0, 4.0\}$ and also run plain ERM, training the same classifier each time and recording test accuracy. What shape do you expect, and what does it tell you about $\alpha$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Fix the network, data, seed, and optimizer; vary only $\alpha$ (with ERM = no mixup as the floor). — _Isolate $\alpha$ as the single variable._
- Recall what $\alpha$ does to $\lambda$: small $\alpha$ → mild mixes near the originals; large $\alpha$ → aggressive 50/50 mixes. — _Connect the knob to the strength of the augmentation._
- Read off test accuracy across the sweep and compare to ERM. — _See whether more mixing keeps helping._

**Answer:** An inverted-U: mixup beats ERM across a wide range, peaking at small-to-moderate $\alpha$ and then degrading as $\alpha$ grows. In our small run (CODEVIZ — our numbers, not the paper's) ERM is 0.935, $\alpha=0.1$ and $\alpha=0.4$ reach ~0.9725, and $\alpha=4.0$ falls back to ~0.954. Too much mixing (every example a hard 50/50 blend) makes the task too hard and underfits — matching the paper's recommendation of $\alpha\in[0.1,0.4]$.

</details>